# Iris Flower Classification: Logistic Regression vs Decision Tree

**Objective:** Build a supervised classification pipeline on the Iris dataset: EDA, feature scaling (`StandardScaler`), 80/20 train/test split, model training (Logistic Regression and Decision Tree), and benchmarking with accuracy, precision, recall and F1-score. Results are visualised with a confusion matrix heatmap and decision boundary plots.

**Pipeline:** Load data → EDA (pairplot) → Split (80/20) → Scale → Train → Evaluate → Visualise

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
from sklearn.inspection import DecisionBoundaryDisplay

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

## 2. Load the Dataset (Pandas)

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"

try:
    df = pd.read_csv(URL)
    print("Loaded dataset from URL")
except Exception as e:
    # Offline fallback: same Iris data bundled with scikit-learn
    from sklearn.datasets import load_iris
    iris = load_iris(as_frame=True)
    df = iris.frame.copy()
    df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
    df["species"] = df["species"].map(dict(enumerate(iris.target_names)))
    print("URL not reachable, loaded Iris from scikit-learn instead:", e)

df.head()

## 3. Exploratory Data Analysis

In [ ]:
print("Shape:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum())
print("\nClass distribution:\n", df["species"].value_counts())
df.describe().T

In [ ]:
# Pairplot: feature relationships coloured by species
g = sns.pairplot(df, hue="species", diag_kind="kde", palette="Set2", corner=False)
g.fig.suptitle("Iris Dataset: Pairplot by Species", y=1.02, fontsize=14)
plt.show()

**Observations**
- *Setosa* is clearly separable from the other two classes, especially on petal length and petal width.
- *Versicolor* and *Virginica* overlap slightly, so those two are where misclassifications are most likely.
- The dataset is perfectly balanced (50 samples per class) and has no missing values.

In [ ]:
# Correlation heatmap of the numeric features
plt.figure(figsize=(6, 4.5))
sns.heatmap(df.drop(columns="species").corr(), annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Feature Correlation Matrix")
plt.show()

## 4. Train/Test Split (80/20) and Feature Scaling

In [ ]:
X = df.drop(columns="species")
y = df["species"]
class_names = sorted(y.unique())

# 80/20 split; stratify keeps class proportions equal in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Fit the scaler on TRAIN data only (avoids data leakage), then transform both sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nScaled train mean (~0):", X_train_scaled.mean(axis=0).round(3))
print("Scaled train std  (~1):", X_train_scaled.std(axis=0).round(3))

## 5. Train the Models

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
dec_tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)

log_reg.fit(X_train_scaled, y_train)
dec_tree.fit(X_train_scaled, y_train)

models = {"Logistic Regression": log_reg, "Decision Tree": dec_tree}
preds = {name: m.predict(X_test_scaled) for name, m in models.items()}
print("Both models trained successfully")

## 6. Evaluation: Accuracy, Precision, Recall, F1-score

In [ ]:
# Macro averaging treats all 3 classes equally (suitable for a balanced dataset)
rows = []
for name, y_pred in preds.items():
    rows.append({
        "Model": name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro"),
        "Recall":    recall_score(y_test, y_pred, average="macro"),
        "F1-score":  f1_score(y_test, y_pred, average="macro"),
    })

results = pd.DataFrame(rows).set_index("Model").round(4)
results

In [ ]:
for name, y_pred in preds.items():
    print("=" * 60)
    print(f"Classification Report: {name}")
    print("=" * 60)
    print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

In [ ]:
# Metric comparison bar chart
ax = results.plot(kind="bar", figsize=(8, 4.5), colormap="Set2", rot=0)
ax.set_ylim(0.8, 1.02)
ax.set_title("Model Benchmark on Test Set")
ax.set_ylabel("Score")
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=8, padding=2)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# 5-fold cross-validation gives a more reliable estimate than a single 30-sample test set
from sklearn.pipeline import make_pipeline

cv_rows = []
for name, model in {"Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)}.items():
    pipe = make_pipeline(StandardScaler(), model)   # scaler inside the pipeline, so no leakage across folds
    scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    cv_rows.append({"Model": name, "CV Mean Accuracy": scores.mean(), "CV Std": scores.std()})

pd.DataFrame(cv_rows).set_index("Model").round(4)

## 7. Confusion Matrix Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for ax, (name, y_pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, y_pred, labels=class_names)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=ax,
                annot_kws={"size": 14})
    ax.set_title(f"{name}\nAccuracy: {accuracy_score(y_test, y_pred):.2%}")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

plt.tight_layout()
plt.show()

## 8. Decision Boundary Visualisation

A decision boundary can only be drawn in 2D, so for this plot the models are retrained on the two most discriminative features, **petal length** and **petal width**. These are the same standardised features used above and are only used for visualisation.

In [ ]:
feat_2d = ["petal_length", "petal_width"]
label_to_int = {c: i for i, c in enumerate(class_names)}

scaler_2d = StandardScaler()
X2_train = scaler_2d.fit_transform(X_train[feat_2d])
X2_test = scaler_2d.transform(X_test[feat_2d])
y2_train = y_train.map(label_to_int).values
y2_test = y_test.map(label_to_int).values

models_2d = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
from matplotlib.colors import ListedColormap
cmap_bg = ListedColormap(plt.cm.Set2.colors[:3])   # one colour per species
for ax, (name, model) in zip(axes, models_2d.items()):
    model.fit(X2_train, y2_train)
    DecisionBoundaryDisplay.from_estimator(
        model, X2_train, response_method="predict", cmap=cmap_bg, alpha=0.6,
        grid_resolution=300, xlabel="Petal length (scaled)", ylabel="Petal width (scaled)", ax=ax)
    scatter = ax.scatter(X2_test[:, 0], X2_test[:, 1], c=y2_test, cmap=cmap_bg,
                         edgecolor="k", s=70, label="Test samples")
    acc = accuracy_score(y2_test, model.predict(X2_test))
    ax.set_title(f"{name}: Decision Boundary\n(2D test accuracy: {acc:.2%})")
    handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=cmap_bg(i),
                          markeredgecolor="k", markersize=9, label=c) for i, c in enumerate(class_names)]
    ax.legend(handles=handles, title="Species", loc="upper left")

plt.tight_layout()
plt.show()

## 9. Decision Tree Structure (Bonus)

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(14, 7))
plot_tree(dec_tree, feature_names=X.columns, class_names=class_names,
          filled=True, rounded=True, fontsize=9)
plt.title("Decision Tree (max_depth=3)")
plt.show()

pd.Series(dec_tree.feature_importances_, index=X.columns).sort_values(ascending=False).round(3)

## 10. Conclusion

| Aspect | Logistic Regression | Decision Tree |
|---|---|---|
| Type | Linear, probabilistic | Non-linear, rule-based |
| Boundary shape | Straight lines | Axis-aligned rectangles |
| Needs scaling? | Yes, helps convergence | No, but harmless |
| Interpretability | Coefficients | Visual if-else rules |

**Key takeaways**
- Both models perform very well on Iris, since the classes are largely separable.
- *Setosa* is classified perfectly; the only confusion occurs between *Versicolor* and *Virginica*.
- Petal length and petal width are the most informative features.
- The test set holds only 30 samples, so a single misclassification shifts accuracy by about 3.3%. Cross-validation confirms the models' real performance.
- `StandardScaler` was fitted on the training data only to prevent data leakage.